In [1]:
# task1_data_collection.py

# Importing required libraries
import requests
import json
import os
import time
from datetime import datetime

# -------------------------------
# API URLs and request header
# -------------------------------
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"
headers = {"User-Agent": "TrendPulse/1.0"}

# -------------------------------
# Category keywords dictionary
# -------------------------------
category_keywords = {
    "technology": ["AI", "software", "tech", "code", "computer", "data", "cloud", "API", "GPU", "LLM"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["NFL", "NBA", "FIFA", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "NASA", "genome"],
    "entertainment": ["movie", "film", "music", "Netflix", "game", "book", "show", "award", "streaming"]
}


# Function to assign category
# based on title keywords
def get_category(title):
    if not title:
        return None

    title_lower = title.lower()

    for category, keywords in category_keywords.items():
        for word in keywords:
            if word.lower() in title_lower:
                return category

    return None



# Function to fetch top story ids
def fetch_top_story_ids():
    try:
        response = requests.get(TOP_STORIES_URL, headers=headers)
        response.raise_for_status()
        return response.json()[:500]
    except Exception as e:
        print(f"Failed to fetch top story IDs: {e}")
        return []



# Function to fetch one story detail
def fetch_story_details(story_id):
    try:
        response = requests.get(ITEM_URL.format(story_id), headers=headers)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Failed to fetch story {story_id}: {e}")
        return None



# Main data collection logic
def collect_trending_stories():
    top_story_ids = fetch_top_story_ids()

    collected_stories = []

    # To track max 25 stories per category
    category_count = {
        "technology": 0,
        "worldnews": 0,
        "sports": 0,
        "science": 0,
        "entertainment": 0
    }

    # Current timestamp to store in collected_at field
    collected_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Loop category by category as asked in task
    for category in category_count.keys():

        print(f"Collecting {category} stories...")

        for story_id in top_story_ids:

            # Skip if already collected 25 for this category
            if category_count[category] >= 25:
                break

            story = fetch_story_details(story_id)

            # If request failed, continue
            if story is None:
                continue

            title = story.get("title", "")

            assigned_category = get_category(title)

            # Only collect if title belongs to current category
            if assigned_category == category:

                story_data = {
                    "post_id": story.get("id"),
                    "title": title,
                    "category": assigned_category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0),
                    "author": story.get("by", "unknown"),
                    "collected_at": collected_time
                }

                collected_stories.append(story_data)
                category_count[category] += 1

        # One sleep after each category loop (as required)
        time.sleep(2)

    return collected_stories


# -------------------------------
# Function to save JSON file
# -------------------------------
def save_to_json(data):
    if not os.path.exists("data"):
        os.makedirs("data")

    file_name = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

    with open(file_name, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4)

    print(f"Collected {len(data)} stories. Saved to {file_name}")


# -------------------------------
# Program execution starts here
# -------------------------------
if __name__ == "__main__":
    final_data = collect_trending_stories()
    save_to_json(final_data)

Collected 85 stories. Saved to data/trends_20260503.json
